##
#**Week 1**

In [1]:
!pip install pandas spacy sentence-transformers faiss-cpu requests
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# --- Imports ---
import requests
import pandas as pd
import time
import json
from datetime import datetime, timezone

# --- Function to fetch posts, with retry protection ---
def fetch_posts(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    all_posts = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        posts = data.get("data", [])
        if not posts:
            print("No more posts — done.")
            break

        all_posts.extend(posts)
        print(f"Posts collected: {len(all_posts)}")

        # Save progress after every batch so nothing is ever lost
        with open("posts_progress.json", "w") as f:
            json.dump(all_posts, f)

        last_created = posts[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(posts) < limit:
            break

    return all_posts

# --- Run it ---
raw_posts = fetch_posts("artificial", "2026-06-01", "2026-06-30")
posts_df = pd.DataFrame(raw_posts)

print(f"\nTOTAL POSTS: {len(raw_posts)}")
print("Posts shape:", posts_df.shape)

Posts collected: 100
Posts collected: 200
Posts collected: 300
Posts collected: 400
Posts collected: 500
Posts collected: 600
Posts collected: 700
Posts collected: 800
Posts collected: 900
Posts collected: 1000
Posts collected: 1100
Posts collected: 1200
Posts collected: 1300
Posts collected: 1400
Posts collected: 1500
Posts collected: 1600
Posts collected: 1700
Posts collected: 1800
Posts collected: 1900
Posts collected: 2000
Posts collected: 2100
Posts collected: 2200
Posts collected: 2300
Posts collected: 2317

TOTAL POSTS: 2317
Posts shape: (2317, 117)


In [3]:

# --- Function to fetch comments, with retry protection ---
def fetch_comments(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"
    all_comments = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        comments = data.get("data", [])
        if not comments:
            print("No more comments — done.")
            break

        all_comments.extend(comments)
        print(f"Comments collected: {len(all_comments)}")

        # Save progress after every batch so nothing is ever lost
        with open("comments_progress.json", "w") as f:
            json.dump(all_comments, f)

        last_created = comments[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(comments) < limit:
            break

    return all_comments

# --- Run it ---
raw_comments = fetch_comments("artificial", "2026-06-01", "2026-06-30")
comments_df = pd.DataFrame(raw_comments)

print(f"\nTOTAL COMMENTS: {len(raw_comments)}")
print("Comments shape:", comments_df.shape)

Comments collected: 100
Comments collected: 200
Comments collected: 300
Comments collected: 400
Comments collected: 500
Comments collected: 600
Comments collected: 700
Comments collected: 800
Comments collected: 900
Comments collected: 1000
Comments collected: 1100
Comments collected: 1200
Comments collected: 1300
Comments collected: 1400
Comments collected: 1500
Comments collected: 1600
Comments collected: 1700
Comments collected: 1800
Comments collected: 1900
Comments collected: 2000
Comments collected: 2100
Comments collected: 2200
Comments collected: 2300
Comments collected: 2400
Comments collected: 2500
Comments collected: 2600
Comments collected: 2700
Comments collected: 2800
Comments collected: 2900
Comments collected: 3000
Comments collected: 3100
Comments collected: 3200
Comments collected: 3300
Comments collected: 3400
Comments collected: 3500
Comments collected: 3600
Comments collected: 3700
Comments collected: 3800
Comments collected: 3900
Comments collected: 4000
Comments 

In [4]:
import re

# --- 1. Combine post title + body into one "text" column ---
posts_df["text"] = posts_df["title"].fillna("") + " " + posts_df["selftext"].fillna("")
comments_df["text"] = comments_df["body"].fillna("")

# --- 2. Remove deleted/removed content ---
# These show up as literal placeholder strings when content is gone
deleted_markers = ["[deleted]", "[removed]"]

posts_clean = posts_df[~posts_df["text"].str.strip().isin(deleted_markers)].copy()
comments_clean = comments_df[~comments_df["text"].str.strip().isin(deleted_markers)].copy()

print(f"Posts after removing deleted/removed: {len(posts_clean)} (was {len(posts_df)})")
print(f"Comments after removing deleted/removed: {len(comments_clean)} (was {len(comments_df)})")

# --- 3. Remove bot comments ---
# AutoModerator is Reddit's most common bot; add more usernames here if you spot others later
bot_authors = ["AutoModerator"]

comments_clean = comments_clean[~comments_clean["author"].isin(bot_authors)].copy()
print(f"Comments after removing bots: {len(comments_clean)}")

# --- 4. Strip URLs and markdown formatting using regex ---
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)          # remove URLs
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)           # **bold** -> bold
    text = re.sub(r"\*(.*?)\*", r"\1", text)               # *italic* -> italic
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)        # [text](link) -> text
    text = re.sub(r"[#>`~]", "", text)                     # remove #, >, `, ~ symbols
    text = re.sub(r"\s+", " ", text).strip()               # collapse extra whitespace
    return text

posts_clean["clean_text"] = posts_clean["text"].apply(clean_text)
comments_clean["clean_text"] = comments_clean["text"].apply(clean_text)

# --- 5. Remove rows that are empty after cleaning ---
posts_clean = posts_clean[posts_clean["clean_text"].str.len() > 0]
comments_clean = comments_clean[comments_clean["clean_text"].str.len() > 0]

# --- 6. Deduplicate ---
posts_clean = posts_clean.drop_duplicates(subset="clean_text")
comments_clean = comments_clean.drop_duplicates(subset="clean_text")

print(f"\nFinal posts: {len(posts_clean)}")
print(f"Final comments: {len(comments_clean)}")

# --- 7. Preview ---
posts_clean[["clean_text"]].head()

Posts after removing deleted/removed: 2317 (was 2317)
Comments after removing deleted/removed: 18791 (was 20447)
Comments after removing bots: 18791

Final posts: 2264
Final comments: 18397


,clean_text
0,HeyGen for AI video side hustles: what it actu...
1,Why do AI agents keep repeating the same brows...
2,I think I broke AI He's been on the same exest...
3,What features do you think are most important ...
4,Ai slop security… [removed]


In [5]:
import spacy
from collections import Counter

# Load the small English model we installed back in Step 1
nlp = spacy.load("en_core_web_sm")

def tokenize(text):
    doc = nlp(text.lower())  # lowercase for consistency
    # keep only alphabetic tokens, remove stopwords (the, is, and...) and punctuation
    tokens = [token.text for token in doc if token.is_alpha and not token.is_stop]
    return tokens

# Apply to a sample first (spaCy is slower on 20k+ rows, sample keeps this fast for now)
sample_comments = comments_clean["clean_text"].sample(min(2000, len(comments_clean)), random_state=42)

all_tokens = []
for text in sample_comments:
    all_tokens.extend(tokenize(text))

print(f"Total tokens (from {len(sample_comments)} sampled comments): {len(all_tokens)}")

# --- Basic corpus stats ---
word_freq = Counter(all_tokens)
top_terms = word_freq.most_common(20)

print("\nTop 20 most common words:")
for word, count in top_terms:
    print(f"{word}: {count}")

Total tokens (from 2000 sampled comments): 43125

Top 20 most common words:
ai: 925
like: 380
people: 344
use: 282
think: 254
model: 230
time: 203
work: 192
actually: 190
way: 180
good: 172
real: 160
data: 160
need: 158
things: 150
models: 150
thing: 145
human: 140
know: 136
right: 125


##
#**Week 2**

In [6]:
!pip install pandas sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [39]:
print("Posts:", posts_clean.shape)
print("Comments:", comments_clean.shape)

Posts: (2264, 119)
Comments: (18397, 77)


In [8]:
import re

def chunk_text(text, max_chunk_size=200, overlap=20):
    """
    Recursively splits text into chunks of roughly max_chunk_size characters.
    Tries paragraph breaks first, then sentences, then words.
    """
    if len(text) <= max_chunk_size:
        return [text] if text.strip() else []

    paragraphs = text.split("\n\n")
    if len(paragraphs) > 1:
        chunks = []
        for para in paragraphs:
            chunks.extend(chunk_text(para, max_chunk_size, overlap))
        return chunks

    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        chunks = []
        current = ""
        for sentence in sentences:
            if len(current) + len(sentence) <= max_chunk_size:
                current += (" " if current else "") + sentence
            else:
                if current:
                    chunks.append(current)
                current = current[-overlap:] + " " + sentence if current else sentence
        if current:
            chunks.append(current)
        return chunks

    words = text.split()
    chunks = []
    current = ""
    for word in words:
        if len(current) + len(word) + 1 <= max_chunk_size:
            current += (" " if current else "") + word
        else:
            chunks.append(current)
            current = word
    if current:
        chunks.append(current)
    return chunks


# ---- Unit tests ----
def test_chunk_text():
    result = chunk_text("Hello world.", max_chunk_size=200)
    assert result == ["Hello world."], f"Test 1 failed: {result}"

    result = chunk_text("", max_chunk_size=200)
    assert result == [], f"Test 2 failed: {result}"

    long_text = "This is a sentence. " * 30
    result = chunk_text(long_text, max_chunk_size=100)
    assert len(result) > 1, f"Test 3 failed: got {len(result)} chunk(s)"

    for chunk in result:
        assert len(chunk) <= 150, f"Test 4 failed: chunk too long ({len(chunk)} chars): {chunk}"

    print("All unit tests passed!")

test_chunk_text()

# ---- Apply chunking to real comments ----

all_chunks = []
chunk_sources = []

for idx, row in comments_clean.iterrows():
    text = row["clean_text"]

    # Skip missing/NaN values entirely
    if pd.isna(text):
        continue

    text = str(text)  # force to string, just in case

    chunks = chunk_text(text, max_chunk_size=300, overlap=30)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_sources.append(row["id"] if "id" in comments_clean.columns else idx)

print(f"\nTotal comments: {len(comments_clean)}")
print(f"Total chunks generated: {len(all_chunks)}")
print(f"\nSample chunk: {all_chunks[0]}")

All unit tests passed!

Total comments: 18397
Total chunks generated: 30508

Sample chunk: That person will just make the model worse than it started


In [11]:
from sentence_transformers import SentenceTransformer
import time

# Load a small, fast, well-regarded embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# To keep this fast for now, let's embed a manageable subset first (5,000 chunks)
# You can scale up to the full 30,507 once we confirm everything works
sample_size = 5000
chunks_sample = all_chunks[:sample_size]

print(f"Embedding {len(chunks_sample)} chunks...")

start_time = time.time()
embeddings = model.encode(chunks_sample, show_progress_bar=True, batch_size=64)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(chunks_sample) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")  # (num_chunks, embedding_dimensions)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 5000 chunks...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]


Done.
Total time: 3.18 seconds
Chunks per second: 1574.26
Embedding shape: (5000, 384)


In [19]:
print(f"Embedding all {len(all_chunks)} chunks...")

start_time = time.time()
embeddings = model.encode(all_chunks, show_progress_bar=True, batch_size=64)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(all_chunks) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")

Embedding all 30508 chunks...


Batches:   0%|          | 0/477 [00:00<?, ?it/s]


Done.
Total time: 21.10 seconds
Chunks per second: 1445.94
Embedding shape: (30508, 384)


In [21]:
import torch

# Confirm we're actually using the GPU (if this prints "cpu", switch runtime type first)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model.half()  # half-precision — faster on GPU, negligible quality loss for this use case

start_time = time.time()
embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=256,          # larger batch = better GPU utilization
    convert_to_numpy=True
)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(all_chunks) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]


Done.
Total time: 9.97 seconds
Chunks per second: 3059.42
Embedding shape: (30508, 384)


In [32]:
# Delete the old collection entirely, then recreate it fresh
chroma_client.delete_collection(name="reddit_comments")
collection = chroma_client.get_or_create_collection(name="reddit_comments")

# Now re-run your ingestion loop as before
chunk_ids = [f"chunk_{i}" for i in range(len(all_chunks))]

print(f"Ingesting {len(all_chunks)} chunks into ChromaDB...")
start_time = time.time()

batch_size = 5000
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids = chunk_ids[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size].tolist()

    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_chunks
    )
    print(f"Ingested {min(i+batch_size, len(all_chunks))}/{len(all_chunks)}")

elapsed = time.time() - start_time
print(f"\nDone. Total ingestion time: {elapsed:.2f} seconds")
print(f"Total items in collection: {collection.count()}")

Ingesting 30508 chunks into ChromaDB...
Ingested 5000/30508
Ingested 10000/30508
Ingested 15000/30508
Ingested 20000/30508
Ingested 25000/30508
Ingested 30000/30508
Ingested 30508/30508

Done. Total ingestion time: 30.32 seconds
Total items in collection: 30508


In [35]:
def semantic_search(query, n_results=5):
    """
    Embeds a query and searches ChromaDB for the most similar chunks.
    Returns results along with how long the search took.
    """
    start_time = time.time()

    # Turn the query into an embedding using the same model as before
    query_embedding = model.encode([query]).tolist()

    # Ask ChromaDB for the n_results closest matches
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    elapsed = time.time() - start_time

    return results, elapsed


# ---- Test it with a few different queries ----
test_queries = [
    "is AI going to replace programmers",
    "AI is making people lose critical thinking skills",
    "how good is Google's AI model",
]

latency_log = []

for query in test_queries:
    results, elapsed = semantic_search(query, n_results=3)
    latency_log.append({"query": query, "latency_seconds": elapsed})

    print(f"\nQuery: \"{query}\"")
    print(f"Latency: {elapsed*1000:.2f} ms")
    print("Top results:")
    for i, doc in enumerate(results["documents"][0]):
        print(f"  {i+1}. {doc[:150]}")  # show first 150 characters

print("\n\n--- Latency Summary ---")
for entry in latency_log:
    print(f"{entry['latency_seconds']*1000:.2f} ms — \"{entry['query']}\"")


Query: "is AI going to replace programmers"
Latency: 17.29 ms
Top results:
  1.  not replacing people with AI. They are using it to handle repetitive tasks, speed up research, write first drafts, and automate routine work. The big
  2. Judging from what you think AI can do, you can probably be replaced by ai too.
  3. Yah. So why you say AI won't replace employees but will replace repetitive tasks ?

Query: "AI is making people lose critical thinking skills"
Latency: 18.33 ms
Top results:
  1. Critical thinking, my man. You have it, AI doesn't
  2. S peaked and began to decline. We are beginning to see anecdotal information showing AI reducing critical thinking skills among users who rely on it t
  3. At least in the US, I think that literacy and critical thinking skills were already on the decline well before generative AI came along. I don't think

Query: "how good is Google's AI model"
Latency: 18.19 ms
Top results:
  1. googles ai is cooked
  2. Ai its a better google people have 

In [38]:
def semantic_search_safe(query, n_results=5):
    """
    Same as semantic_search, but handles edge cases gracefully
    instead of crashing.
    """
    # Edge case 1: empty or whitespace-only query
    if not query or not query.strip():
        return {"error": "Query cannot be empty"}, 0

    # Edge case 2: absurdly long query (cap it so embedding doesn't choke)
    max_query_length = 1000
    if len(query) > max_query_length:
        query = query[:max_query_length]

    # Edge case 3: asking for more results than exist in the collection
    available = collection.count()
    n_results = min(n_results, available)

    start_time = time.time()
    try:
        query_embedding = model.encode([query]).tolist()
        results = collection.query(query_embeddings=query_embedding, n_results=n_results)
    except Exception as e:
        return {"error": str(e)}, time.time() - start_time

    elapsed = time.time() - start_time
    return results, elapsed


# ---- Test the edge cases ----
print("Test: empty query")
result, elapsed = semantic_search_safe("")
print(result)

print("\nTest: whitespace-only query")
result, elapsed = semantic_search_safe("   ")
print(result)

print("\nTest: requesting way more results than exist")
result, elapsed = semantic_search_safe("AI ethics", n_results=999999)
print(f"Got {len(result['documents'][0])} results (collection only has {collection.count()} items)")

print("\nTest: normal query still works")
result, elapsed = semantic_search_safe("AI ethics")
print(f"Got {len(result['documents'][0])} results, latency {elapsed*1000:.2f}ms")

Test: empty query
{'error': 'Query cannot be empty'}

Test: whitespace-only query
{'error': 'Query cannot be empty'}

Test: requesting way more results than exist
Got 30507 results (collection only has 30508 items)

Test: normal query still works
Got 5 results, latency 9.59ms


## Chunking Strategy

To prepare comments for embedding, I implemented a **recursive text-chunking** approach:

1. Try splitting on paragraph breaks first (most natural break point)
2. If a piece is still too long, fall back to splitting on sentence boundaries
3. If a single sentence is still too long, fall back to splitting on word boundaries

**Parameters used:** max chunk size of 300 characters, with a 30-character overlap between
consecutive chunks so context isn't abruptly lost at a chunk boundary.

**Why these choices:** Reddit comments are mostly short, so 300 characters keeps most
comments as a single chunk, while longer, multi-topic comments get split into more
focused pieces. The overlap preserves a bit of context across chunk boundaries without
meaningfully increasing the total chunk count.

**Verified with unit tests** covering: short text (returns unmodified), empty text
(returns empty list), long text (splits into multiple chunks), and chunk size limits
(no chunk wildly exceeds the target size).

**Result on real data:** 18,397 cleaned comments → 30,507 chunks (~1.7 chunks/comment).